In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langchain_community.embeddings import HuggingFaceEmbeddings

# Setup Google API Key for the LLM
os.environ["GOOGLE_API_KEY"] = "my_api_key" 

# Load the text document
print("Loading the text file...")
loader = TextLoader("company_data.txt")  
docs = loader.load()

# Setup Local Embeddings
# Using HuggingFace here so we can run embeddings locally without API limits
print("Building local FAISS vector database... (This might take 10-15 seconds on the first run)")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(docs, embeddings)

# Create a retriever to search the database
retriever = vector_db.as_retriever()

@tool
def company_data_search(query: str) -> str:
    """Use this tool to search for sales data, Q1, Q2 numbers, or any company information in the provided text file."""
    results = retriever.invoke(query)
    # Join the search results into a single string for the LLM
    return "\n\n".join([doc.page_content for doc in results])

rag_tool = company_data_search

print("\nSprint 1 Complete: Local RAG Tool is ready!")

c:\Users\urvah\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading the text file...


RuntimeError: Error loading company_data.txt

In [9]:
from langchain_core.tools import tool

# Custom Math Tool (Statistical Engine)
@tool
def calculate_regression_slope(x_values: list[float], y_values: list[float]) -> float:
    """Use this tool to calculate the linear regression slope (trend) given two lists of numbers (X and Y).
    If slope is positive, the trend is increasing. If negative, it is decreasing.
    """
    n = len(x_values)
    if n == 0 or n != len(y_values):
        return 0.0
    
    # Core Statistical Formulas
    sum_x = sum(x_values)
    sum_y = sum(y_values)
    sum_xy = sum(x * y for x, y in zip(x_values, y_values))
    sum_xx = sum(x * x for x in x_values)
    
    denominator = (n * sum_xx - sum_x**2)
    if denominator == 0: 
        return 0.0
    
    # Calculating the slope (b or m)
    slope = (n * sum_xy - sum_x * sum_y) / denominator
    
    return round(slope, 4) # Round to 4 decimal places for accuracy

# Assign to a variable to easily pass it to the agent later
math_tool = calculate_regression_slope

print("✅ Sprint 2 Complete: Statistical Math Tool is ready!")

✅ Sprint 2 Complete: Statistical Math Tool is ready!


In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# 1. Initialize the LLM (The Main Brain)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 2. Define the agent's toolbelt (Our two custom tools)
tools = [rag_tool, math_tool]

# 3. Assemble the Agent using LangGraph
# This approach is much more robust and manageable than the legacy AgentExecutor
smart_agent = create_react_agent(llm, tools)

# THE ULTIMATE TEST
print("\nStarting the LangGraph Agent execution...\n")

user_query = """
First, use the company data search tool to find out the total sales for Q1, Q2, Q3, and Q4. 
After that, use the math tool to calculate the linear regression slope on those four values. 
Assume the X values are [1, 2, 3, 4] (where 1=Q1, 2=Q2, etc.), and the Y values are the four sales figures you extracted.
"""

# Pass the prompt to the agent (LangGraph expects a list of message objects)
response = smart_agent.invoke({"messages": [HumanMessage(content=user_query)]})

print("FINAL ANSWER FROM AGENT:")
# LangGraph returns the entire chat history, so we just extract the final message
print(response["messages"][-1].content)

C:\Users\urvah\AppData\Local\Temp\ipykernel_32252\23813011.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  smart_agent = create_react_agent(llm, tools)



Starting the LangGraph Agent execution...



ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 15.567179945s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT